# Manutenção preditiva em linha de massas longas (KDD + classificação)

Trabalho de conclusão de curso (Engenharia de Produção, UCS, 2025): aplicação das etapas do KDD a registros de paradas de linha de um sistema MES de uma fábrica de alimentos.

A partir dos eventos de parada, o notebook calcula o **tempo até a próxima manutenção** e o usa como alvo de dois problemas de classificação:

1. **5 janelas** de tempo (0-12h, 12-24h, 24-36h, 36-48h, >48h): Regressão Logística Multinomial, Random Forest e XGBoost
2. **Risco binário** (manutenção em até 24h ou não): Regressão Logística e XGBoost, avaliados com ROC-AUC

> **Os dados em `data/paradas_exemplo.xlsx` são sintéticos**, gerados só para demonstrar o código. Os dados reais do estudo não são públicos.

## 1. Preparação dos dados

In [ ]:
# LINHA DE MASSAS LONGAS – Função de preparação
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def prep_longa2(xls_path="data/paradas_exemplo.xlsx", focus_station=None):
    df = pd.read_excel(Path(xls_path), "Paradas").copy()
    if focus_station:
        df = df[df["ESTACAO"]==focus_station].copy()

    df["DATA_OCORRENCIA_INICIAL"] = pd.to_datetime(df["DATA_OCORRENCIA_INICIAL"], errors="coerce")
    df = df.sort_values("DATA_OCORRENCIA_INICIAL").reset_index(drop=True)
    df["IS_MANUTENCAO"] = df["ULTIMA_DESCRICAO"].fillna("").str.upper().str.contains("MANUTEN")
    manut = df.loc[df["IS_MANUTENCAO"], "DATA_OCORRENCIA_INICIAL"]

    def hrs_next(t):
        fut = manut[manut > t]
        if len(fut)==0: return np.nan
        return (fut.min()-t).total_seconds()/3600

    df["HRS_PROX_MANUT"] = df["DATA_OCORRENCIA_INICIAL"].apply(hrs_next)
    df = df.dropna(subset=["HRS_PROX_MANUT"]).copy()

    bins = [0,12,24,36,48,np.inf]
    labels = ["0-12h","12-24h","24-36h","36-48h",">48h"]
    df["CLASSE_RISCO"] = pd.cut(df["HRS_PROX_MANUT"], bins=bins, labels=labels, include_lowest=True, right=True)
    df["ALTO_RISCO_24H"] = (df["HRS_PROX_MANUT"] <= 24).astype(int)

    df["HORA"] = df["DATA_OCORRENCIA_INICIAL"].dt.hour
    df["MES"] = df["DATA_OCORRENCIA_INICIAL"].dt.month
    df["DIA_SEMANA"] = df["DATA_OCORRENCIA_INICIAL"].dt.dayofweek
    df["TOTAL_TEMPO_SEGUNDOS"] = pd.to_numeric(df["TOTAL_TEMPO_SEGUNDOS"], errors="coerce").fillna(0)

    X_cat = pd.get_dummies(df[["ESTACAO","ULTIMA_DESCRICAO","TURNO"]].fillna("DESCONHECIDO").astype(str), drop_first=True)
    X_num = df[["TOTAL_TEMPO_SEGUNDOS","HORA","MES","DIA_SEMANA"]].copy()
    X_all = pd.concat([X_num, X_cat], axis=1)

    return df, X_all


## 2. Classificação em 5 janelas de tempo

In [ ]:
# LINHA DE MASSAS LONGAS – Logística Multinomial (5 janelas)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

df, X_all = prep_longa2()  # ou prep_longa2(focus_station="ESTACAO_A")
y = df["CLASSE_RISCO"]
Xtr, Xte, ytr, yte = train_test_split(X_all, y, test_size=0.30, random_state=42, stratify=y)

scaler = StandardScaler(with_mean=False)
Xtr_s = scaler.fit_transform(Xtr); Xte_s = scaler.transform(Xte)

lr = LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs")
lr.fit(Xtr_s, ytr)
yp = lr.predict(Xte_s)

print("Logística Multinomial (5 janelas)")
print(classification_report(yte, yp, digits=3))

ConfusionMatrixDisplay.from_predictions(yte, yp, cmap="Blues", xticks_rotation=45)
plt.title("Linha de massas longas — Matriz (Logística Multinomial)")
plt.tight_layout(); plt.show()


Princípio: GLM multiclasse; fronteiras lineares entre classes.

Bom quando: você quer explicabilidade e classes bem separáveis linearmente.

Limitações: sofre em não linearidades e classes raras; pode confundir janelas intermediárias.

In [ ]:
# LINHA DE MASSAS LONGAS – RandomForestClassifier (5 janelas)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

df, X_all = prep_longa2()
y = df["CLASSE_RISCO"]
Xtr, Xte, ytr, yte = train_test_split(X_all, y, test_size=0.30, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=42)
rf.fit(Xtr, ytr)
yp = rf.predict(Xte)

print("RandomForest (5 janelas)")
print(classification_report(yte, yp, digits=3))

ConfusionMatrixDisplay.from_predictions(yte, yp, cmap="Purples", xticks_rotation=45)
plt.title("Linha de massas longas — Matriz (RandomForest)")
plt.tight_layout(); plt.show()


Princípio: ensemble de árvores (bagging); não linear e robusto.

Bom quando: há padrões complexos; costuma melhorar F1 nas classes majoritárias (ex.: 0–12h).

Limitações: classes muito raras continuam difíceis; interpretabilidade moderada.

In [ ]:
# LINHA DE MASSAS LONGAS — XGBoostClassifier (5 janelas) COM ENCODER E PESOS DE CLASSE
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight

# Usa a função prep_longa2 já enviada antes; se não estiver na sessão, cole-a e rode.
df, X_all = prep_longa2()  # ou prep_longa2(focus_station="ESTACAO_A")

# 1) Mapeamento estável de classes (strings -> ints)
class_order = ["0-12h","12-24h","24-36h","36-48h",">48h"]
mapping = {c:i for i, c in enumerate(class_order)}

y_str = df["CLASSE_RISCO"].astype(str)
y_enc = y_str.map(mapping).astype(int)  # agora rótulos 0..4

# 2) Split estratificado
Xtr, Xte, ytr, yte = train_test_split(
    X_all, y_enc, test_size=0.30, random_state=42, stratify=y_enc
)

# 3) (Opcional) Pesos por classe para desbalanceamento
USE_CLASS_WEIGHTS = True
fit_kwargs = {}
if USE_CLASS_WEIGHTS:
    classes = np.arange(len(class_order))
    cls_weights = compute_class_weight(class_weight="balanced", classes=classes, y=ytr)
    sample_weight = np.take(cls_weights, ytr)  # peso por amostra
    fit_kwargs["sample_weight"] = sample_weight

# 4) Modelo XGBoost
xgbc = XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.07,
    subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
    objective="multi:softprob", num_class=len(class_order),
    random_state=42, tree_method="hist", eval_metric="mlogloss"
)
xgbc.fit(Xtr, ytr, **fit_kwargs)

# 5) Avaliação
yp = xgbc.predict(Xte)

print("XGBoost (5 janelas) — relatório por classe:")
print(classification_report(
    yte, yp,
    labels=list(range(len(class_order))),
    target_names=class_order,
    digits=3
))

# Matriz de confusão com rótulos legíveis
ConfusionMatrixDisplay.from_predictions(
    yte, yp, labels=list(range(len(class_order))),
    display_labels=class_order, cmap="Greens", xticks_rotation=45
)
plt.title("Linha de massas longas — Matriz de Confusão (XGBoost 5 janelas)")
plt.tight_layout(); plt.show()

# 6) Gráfico de recall por classe (ajuda a ver desequilíbrio)
rep = classification_report(
    yte, yp, labels=list(range(len(class_order))),
    target_names=class_order, digits=3, output_dict=True
)
recall_por_classe = [rep[c]["recall"] for c in class_order]

plt.figure(figsize=(8,4))
plt.bar(class_order, recall_por_classe)
plt.ylabel("Recall")
plt.title("Linha de massas longas — Recall por classe (XGBoost 5 janelas)")
plt.ylim(0,1)
plt.tight_layout(); plt.show()

Princípio: boosting de árvores; aprende erros residuais iterativamente.

Bom quando: quer maior acurácia em tabular e fronteiras complexas.

Limitações: pode superajustar em classes raras; tuning sensível.

## 3. Classificação binária: manutenção em até 24h

In [ ]:
# LINHA DE MASSAS LONGAS – Logística Binária (≤24h) + ROC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

df, X_all = prep_longa2()
y = df["ALTO_RISCO_24H"]
Xtr, Xte, ytr, yte = train_test_split(X_all, y, test_size=0.30, random_state=42, stratify=y)

scaler = StandardScaler(with_mean=False)
Xtr_s = scaler.fit_transform(Xtr); Xte_s = scaler.transform(Xte)

lr = LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs")
lr.fit(Xtr_s, ytr)
yp = lr.predict(Xte_s); proba = lr.predict_proba(Xte_s)[:,1]

print("Logística Binária (≤24h)")
print(classification_report(yte, yp, digits=3))
print("ROC-AUC:", roc_auc_score(yte, proba))

ConfusionMatrixDisplay.from_predictions(yte, yp, cmap="Greys")
plt.title("Linha de massas longas — Matriz (Logística Binária)"); plt.tight_layout(); plt.show()

RocCurveDisplay.from_predictions(yte, proba)
plt.title("Linha de massas longas — Curva ROC (Logística Binária)"); plt.tight_layout(); plt.show()


Princípio: classificação 0/1 com fronteira linear; métrica-chave ROC-AUC.

Bom quando: decisão operacional simples (risco ≤24h: sim/não); interpretável.

Limitações: perde granularidade de janelas; fronteira linear pode submodelar.

In [ ]:
# LINHA DE MASSAS LONGAS – XGBoost (≤24h) + ROC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

df, X_all = prep_longa2()
y = df["ALTO_RISCO_24H"]
Xtr, Xte, ytr, yte = train_test_split(X_all, y, test_size=0.30, random_state=42, stratify=y)

xgb_b = XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.07,
    subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
    objective="binary:logistic", random_state=42, tree_method="hist"
)
xgb_b.fit(Xtr, ytr)
yp = xgb_b.predict(Xte); proba = xgb_b.predict_proba(Xte)[:,1]

print("XGBoost (≤24h)")
print(classification_report(yte, yp, digits=3))
print("ROC-AUC:", roc_auc_score(yte, proba))

ConfusionMatrixDisplay.from_predictions(yte, yp, cmap="Reds")
plt.title("Linha de massas longas — Matriz (XGB Binário)"); plt.tight_layout(); plt.show()

RocCurveDisplay.from_predictions(yte, proba)
plt.title("Linha de massas longas — Curva ROC (XGB Binário)"); plt.tight_layout(); plt.show()


Princípio: boosting para separar positivo (≤24h) vs negativo; mede-se ROC-AUC e F1 da classe 1.

Bom quando: queremos maior poder discriminativo e relações não lineares.

Limitações: tuning; menos interpretável (compense com importância de features se quiser)

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

fpr, tpr, _ = roc_curve(yte, proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1])
plt.xlabel("Taxa de falsos positivos")
plt.ylabel("Taxa de verdadeiros positivos")
plt.title("Linha de massas longas — Curva ROC (Manutenção em até 24h)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()